# 03 · Cross-bank graph & ring detection
Builds the transaction graph, runs ring detection from each complaint seed, and compares siloed (B0) vs Trail. Mirrors `scripts/run_evaluation.py`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "backend"))   # run from repo root or notebooks/
import pandas as pd
import matplotlib.pyplot as plt
from app.simulator import generate_dataset
from app.detection.features import FEATURE_NAMES, compute_features_batch
from app.detection.scorer import RiskScorer
import networkx as nx
from app.detection.lineage import build_transaction_graph
from app.detection.graph import detect_ring_from_transactions
from app.detection.evaluation import run_evaluation, EvalConfig

In [ ]:
data = generate_dataset(seed=42)
txs = data["transactions"]
G = build_transaction_graph(txs)
print(G.number_of_nodes(), "accounts,", G.number_of_edges(), "transfers")
print(pd.Series({m["pattern"]: 0 for m in data["ring_meta"]}).index.tolist())

## One ring of each pattern, coloured by institution

In [ ]:
colors = {"BANK_A": "#4C78A8", "BANK_B": "#F58518", "BANK_C": "#54A24B", "WALLET_W": "#B279A2", "UNKNOWN": "#999"}
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, pat in zip(axes, ("chain", "fanout", "fanin", "cross_bank_hop")):
    m = next(m for m in data["ring_meta"] if m["pattern"] == pat)
    ring = detect_ring_from_transactions(txs, m["seed"])
    H = nx.DiGraph(); inst = {h.hop_index: h for h in ring.hops}
    for h in ring.hops: H.add_node(h.hop_index, inst=h.institution)
    H.add_edges_from(ring.edges)
    pos = nx.spring_layout(H, seed=3)
    nx.draw(H, pos, ax=ax, node_color=[colors.get(H.nodes[n]["inst"], "#999") for n in H], with_labels=False, node_size=300, arrows=True)
    ax.set_title(f"{pat}\n{ring.n_hops} hops · {len(ring.institutions)} institutions · {ring.recommended_action.value}")
plt.tight_layout(); plt.show()

## What would Bank A alone have seen?

In [ ]:
m = next(m for m in data["ring_meta"] if m["pattern"] == "cross_bank_hop")
ring = detect_ring_from_transactions(txs, m["seed"])
bank_a = [t for t in txs if "BANK_A" in (t.source_institution, t.destination_institution)]
Ga = build_transaction_graph(bank_a)
reach_a = nx.descendants(Ga, m["seed"]) | {m["seed"]} if m["seed"] in Ga else set()
print("Trail ring accounts:", ring.n_hops, "| institutions:", ring.institutions)
print("Accounts reachable from the seed using only Bank A's records:", len(reach_a))

## Siloed vs Trail

In [ ]:
rep = run_evaluation(data)
from IPython.display import Markdown
Markdown(rep.to_markdown())

## Robustness: fewer complaints

In [ ]:
rows = []
for rate in (1.0, 0.5, 0.25):
    r = run_evaluation(data, EvalConfig(complaint_rate=rate, sweep_participation=False))
    rows.append({"complaint_rate": rate, "rings_identified": r.trail["ring_identified_rate"], "harm_minutes_per_ring": r.trail["harm_minutes_per_ring"]})
pd.DataFrame(rows)